In [1]:
import os, shutil, math, glob

parent_dir = '/storage/home/hcoda1/5/sjamdade3/scratch/August_13_2023/water_screening/MD_MC/Manuscript_Final/Defects/'

path_general_files = '/storage/home/hcoda1/5/sjamdade3/scratch/August_13_2023/water_screening/MD_MC/Manuscript_Final/Defects/SOP_files/'

MOF_type = 'defects/'

In [2]:
path_CIFs = os.path.join(parent_dir, 'MOFs', str(MOF_type))

path_MD_data = os.path.join(path_CIFs, 'MD_data')

path_GCMC = os.path.join(path_CIFs, 'GCMC/')

os.mkdir(path_GCMC)

path_GCMC_input_files = os.path.join(path_general_files, 'GCMC_input_files/')

files_GCMC_input = os.listdir(path_GCMC_input_files)

path_python_files = os.path.join(path_general_files, 'python_files/')

path_MOF_NVT_MC = os.path.join(path_CIFs, 'NVT_MC/')

path_NVT_MC_1 = os.path.join(path_CIFs, 'PRE_NVT_MC_1/')

### Creating GCMC folders for each MOF at 7 pressures

MOFs = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 19, 21, 22]

for mof in MOFs:
    
    path_GCMC_MOF = os.path.join(path_GCMC, str(mof))
    os.mkdir(path_GCMC_MOF)
    
    P=[410, 820, 1230, 2050, 2870, 3280, 4100]
    
    for count, value in enumerate(P,start=1):
        path_pressure = os.path.join(path_GCMC_MOF, str(count))
        os.mkdir(path_pressure)
        
        path_RestartInitial = os.path.join(path_pressure, 'RestartInitial')
        os.mkdir(path_RestartInitial)
        path_RestartInitial_System_0 = os.path.join(path_RestartInitial, 'System_0')
        os.mkdir(path_RestartInitial_System_0)
        
        for file in os.listdir(os.path.join(path_MD_data, str(mof))):

            if not file.startswith('data.') and file.endswith('.cif'):
                
                f_CIF_file = file
                
        shutil.copy(os.path.join(path_MD_data, str(mof), str(f_CIF_file)),  path_pressure)
        
        path_NVT_MC_2_MOF_restart = os.path.join(path_NVT_MC_1, str(mof), 'Using_Restart', 'Restart', 'System_0')
        
        for file in os.listdir(path_NVT_MC_2_MOF_restart):

            if file.startswith('restart_'):
                f_restart_file = file        
            shutil.copy(os.path.join(path_NVT_MC_2_MOF_restart, str(f_restart_file)), path_RestartInitial_System_0)
               
        for f2 in files_GCMC_input:
            shutil.copy(os.path.join(path_GCMC_input_files,str(f2)), path_pressure)
                    
                    
        fin = open(''+str(path_pressure)+'/simulation.input', "rt")

        data1 = fin.read()

        MOF_Name = str((os.path.splitext(str(f_CIF_file))[0]))

        data1 = data1.replace('MOF', str(MOF_Name))

        data1 = data1.replace('myp', str(value))

        fin.close()

        fin = open(''+str(path_pressure)+'/simulation.input', "wt")

        fin.write(data1)

        fin.close() 
        
        for f_restart in os.listdir(os.path.join(path_pressure, 'RestartInitial', 'System_0')):
            rename_restart = 'restart_'+str(MOF_Name)+'_1.1.1_298.000000_'+str(value)
            os.rename(os.path.join(path_pressure, 'RestartInitial', 'System_0', str(f_restart)), os.path.join(path_pressure, 'RestartInitial', 'System_0', str(rename_restart)))

    shutil.copy(path_general_files+'Job_Array.sh', path_GCMC_MOF)



shutil.copy(path_general_files+'script_batch_GCMC', path_GCMC)

fin = open(''+str(path_GCMC)+'/script_batch_GCMC', "rt")

data1 = fin.read()

data1 = data1.replace('NO_MOF', str(len(MOFs)))
data1 = data1.replace('path_GCMC', str(path_GCMC))

fin.close()

fin = open(''+str(path_GCMC)+'/script_batch_GCMC', "wt")

fin.write(data1)

fin.close() 
